# KGSS 변수 확인 노트북

이 노트북은 KGSS 2003-2025 누적 자료에서 성공 인식 분석에 사용할 주요 변수가 존재하는지 확인하기 위한 스타터 노트북입니다.

원자료(`.sav`)는 Git에 포함하지 않으며, 이 노트북은 로컬 환경에서 `data/raw/kor_data_CUM0074_V2.sav` 파일이 있을 때만 실행됩니다.

## 1. 필요한 라이브러리 불러오기

데이터 처리에는 `pandas`와 `numpy`, SPSS `.sav` 파일 읽기에는 `pyreadstat`, 간단한 시각화 준비를 위해 `matplotlib.pyplot`을 사용합니다.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import pyreadstat
import matplotlib.pyplot as plt

## 2. KGSS 원자료 파일 경로 지정하기

노트북은 `notebooks` 폴더 안에 있으므로, 원자료는 상대 경로 `../data/raw/kor_data_CUM0074_V2.sav`에서 찾습니다. 원자료는 Git에 올리지 않는 파일이므로, 로컬 컴퓨터에 파일이 있을 때만 다음 단계가 실행됩니다.

In [2]:
data_path = Path("../data/raw/kor_data_CUM0074_V2.sav")

if not data_path.exists():
    raise FileNotFoundError(
        f"KGSS raw data file was not found at: {data_path}\n"
        "Place the local .sav file in data/raw and run this notebook again."
    )

data_path

PosixPath('../data/raw/kor_data_CUM0074_V2.sav')

## 3. SPSS `.sav` 파일 읽기

`pyreadstat.read_sav()`를 사용해 KGSS 원자료를 불러옵니다. `df`에는 데이터가, `meta`에는 변수 라벨 등 메타데이터가 저장됩니다.

In [3]:
df, meta = pyreadstat.read_sav(data_path)

print("Data loaded successfully.")

Data loaded successfully.


## 4. 데이터의 기본 구조 확인하기

데이터의 행과 열 개수, 전체 변수 수, 처음 다섯 행을 확인합니다. 이 단계는 파일이 정상적으로 읽혔는지 빠르게 점검하는 용도입니다.

In [4]:
print("Data shape:", df.shape)
print("Number of columns:", len(df.columns))

df.head()

Data shape: (23282, 3491)
Number of columns: 3491


,YEAR,RESPID,YRRESPID,FINALWT,SEX,AGE,MARITAL,EMPLY,WHYNOE,EDUC,...,HHDNO,HOMPOP,SEPAPOP,UNRELAT,COHAP,REGION,URBAN,SAMPLEAB,INTDATM,INTDATD
0,2003.0,102.0,200300102.0,0.437465,1.0,31.0,5.0,1.0,-1.0,3.0,...,2.0,1.0,0.0,1.0,-1.0,1.0,1.0,-1.0,8.0,8.0
1,2003.0,104.0,200300104.0,0.437465,1.0,30.0,5.0,1.0,-1.0,3.0,...,4.0,1.0,0.0,0.0,-1.0,1.0,1.0,-1.0,8.0,7.0
2,2003.0,105.0,200300105.0,0.372542,2.0,30.0,5.0,1.0,-1.0,4.0,...,5.0,1.0,0.0,0.0,-1.0,1.0,1.0,-1.0,8.0,8.0
3,2003.0,106.0,200300106.0,0.745084,2.0,34.0,5.0,1.0,-1.0,6.0,...,6.0,2.0,0.0,0.0,-1.0,1.0,1.0,-1.0,8.0,8.0
4,2003.0,108.0,200300108.0,0.874929,1.0,34.0,1.0,1.0,-1.0,5.0,...,8.0,2.0,0.0,0.0,-1.0,1.0,1.0,-1.0,8.0,8.0


## 5. 분석 후보 변수 목록 만들기

분석에 사용할 후보 변수를 목록으로 정리합니다. 성공 인식 변수와 기본 분석에 필요한 연도, 나이, 가중치 변수가 포함되어 있습니다.

In [5]:
variables_to_check = [
    "YEAR",
    "AGE",
    "FINALWT",
    "SUCDEFRT",
    "SUCDWLTH",
    "SUCDPAED",
    "SUCDKNOW",
    "KIDSOL06",
]

success_variables = [
    "SUCDEFRT",
    "SUCDWLTH",
    "SUCDPAED",
    "SUCDKNOW",
]

child_success_variable = "KIDSOL06"

## 6. 변수 존재 여부 확인하기

후보 변수가 KGSS 데이터 안에 실제로 있는지 확인합니다. 변수명이 다르거나 해당 연도에 포함되지 않은 변수는 `exists` 값이 `False`로 표시됩니다.

In [6]:
variable_check = pd.DataFrame(
    {
        "variable": variables_to_check,
        "exists": [variable in df.columns for variable in variables_to_check],
    }
)

variable_check

,variable,exists
0,YEAR,True
1,AGE,True
2,FINALWT,True
3,SUCDEFRT,True
4,SUCDWLTH,True
5,SUCDPAED,True
6,SUCDKNOW,True
7,KIDSOL06,True


## 7. 각 변수의 값 분포 확인하기

각 후보 변수에 대해 값별 빈도를 출력합니다. 결측값도 함께 확인하기 위해 `dropna=False` 옵션을 사용합니다.

In [7]:
for variable in variables_to_check:
    print("=" * 80)
    print(f"Value counts for {variable}")
    print("=" * 80)

    if variable not in df.columns:
        print(f"{variable} does not exist in the dataset.\n")
        continue

    print(df[variable].value_counts(dropna=False).sort_index())
    print()

Value counts for YEAR
YEAR
2003.0    1315
2004.0    1312
2005.0    1613
2006.0    1605
2007.0    1431
2008.0    1508
2009.0    1599
2010.0    1576
2011.0    1535
2012.0    1396
2013.0    1294
2014.0    1370
2016.0    1051
2018.0    1031
2021.0    1205
2023.0    1230
2025.0    1211
Name: count, dtype: int64

Value counts for AGE
AGE
-8.0      21
 18.0    244
 19.0    328
 20.0    328
 21.0    287
        ... 
 93.0      6
 94.0      3
 95.0      2
 97.0      1
 99.0      1
Name: count, Length: 81, dtype: int64

Value counts for FINALWT
FINALWT
0.203211     1
0.208391     3
0.210053     1
0.214734     1
0.223605    10
            ..
4.591092     1
4.618490     1
4.769760     1
4.934609     2
5.287165     3
Name: count, Length: 5229, dtype: int64

Value counts for SUCDEFRT
SUCDEFRT
-8.0        2
-1.0    16667
 1.0     2518
 2.0     2841
 3.0     1076
 4.0      160
 5.0       18
Name: count, dtype: int64

Value counts for SUCDWLTH
SUCDWLTH
-8.0        7
-1.0    16667
 1.0     1248
 2.0    

## 8. 성공 인식 변수의 연도별 결측이 아닌 값 수 확인하기

`YEAR`를 기준으로 성공 인식 변수들의 결측이 아닌 응답 수를 계산합니다. 이를 통해 어떤 연도에 어떤 성공 인식 문항이 포함되었는지 확인할 수 있습니다.

In [8]:
if "YEAR" not in df.columns:
    print("YEAR variable does not exist in the dataset.")
else:
    available_success_variables = [
        variable for variable in success_variables if variable in df.columns
    ]

    if not available_success_variables:
        print("None of the success perception variables exist in the dataset.")
    else:
        yearly_success_counts = (
            df.groupby("YEAR")[available_success_variables]
            .count()
            .sort_index()
        )
        display(yearly_success_counts)

,SUCDEFRT,SUCDWLTH,SUCDPAED,SUCDKNOW
YEAR,,,,
2003.0,1315,1315,1315,1315
2004.0,1312,1312,1312,1312
2005.0,1613,1613,1613,1613
2006.0,1605,1605,1605,1605
2007.0,1431,1431,1431,1431
2008.0,1508,1508,1508,1508
2009.0,1599,1599,1599,1599
2010.0,1576,1576,1576,1576
2011.0,1535,1535,1535,1535


## 9. `KIDSOL06` 변수의 연도별 결측이 아닌 값 수 확인하기

자녀 관련 성공 기대 문항으로 사용할 수 있는 `KIDSOL06` 변수의 연도별 결측이 아닌 응답 수를 확인합니다.

In [9]:
if "YEAR" not in df.columns:
    print("YEAR variable does not exist in the dataset.")
elif child_success_variable not in df.columns:
    print(f"{child_success_variable} does not exist in the dataset.")
else:
    yearly_child_success_counts = (
        df.groupby("YEAR")[child_success_variable]
        .count()
        .sort_index()
        .rename("non_null_count")
        .to_frame()
    )
    display(yearly_child_success_counts)

,non_null_count
YEAR,
2003.0,1315
2004.0,1312
2005.0,1613
2006.0,1605
2007.0,1431
2008.0,1508
2009.0,1599
2010.0,1576
2011.0,1535


## 10. 집계 결과 저장 및 응답 라벨 확인하기

이 단계에서는 개인 단위 자료를 저장하지 않고, 앞에서 만든 변수 확인표와 연도별 집계표만 `outputs/tables` 폴더에 CSV 파일로 저장합니다. 또한 성공 인식 변수는 유효 응답값 1-5만 따로 집계하고, `KIDSOL06`의 값 라벨을 메타데이터에서 확인합니다.

In [10]:
output_table_dir = Path("../outputs/tables")
output_table_dir.mkdir(parents=True, exist_ok=True)

variable_check.to_csv(
    output_table_dir / "01_variable_check.csv",
    index=False,
)

if "yearly_success_counts" in globals():
    yearly_success_counts.to_csv(
        output_table_dir / "01_yearly_success_counts_raw_count.csv",
        index=True,
    )
else:
    print("yearly_success_counts was not created, so it was not saved.")

if "yearly_child_success_counts" in globals():
    yearly_child_success_counts.to_csv(
        output_table_dir / "01_yearly_child_success_counts_raw_count.csv",
        index=True,
    )
else:
    print("yearly_child_success_counts was not created, so it was not saved.")

if "YEAR" not in df.columns:
    print("YEAR variable does not exist, so valid response counts were not saved.")
else:
    available_success_variables = [
        variable for variable in success_variables if variable in df.columns
    ]

    if not available_success_variables:
        print("No success perception variables exist, so valid response counts were not saved.")
    else:
        yearly_success_counts_valid_1_5 = (
            df[available_success_variables]
            .isin([1, 2, 3, 4, 5])
            .assign(YEAR=df["YEAR"])
            .groupby("YEAR")[available_success_variables]
            .sum()
            .astype(int)
            .sort_index()
        )
        yearly_success_counts_valid_1_5.to_csv(
            output_table_dir / "01_yearly_success_counts_valid_1_5.csv",
            index=True,
        )
        display(yearly_success_counts_valid_1_5)

kidsol06_labels = meta.variable_value_labels.get("KIDSOL06", {})

print("KIDSOL06 value labels")
if kidsol06_labels:
    for value, label in kidsol06_labels.items():
        print(f"{value}: {label}")
else:
    print("No value labels found for KIDSOL06 in pyreadstat metadata.")

print(f"Aggregated summary tables saved to: {output_table_dir}")

,SUCDEFRT,SUCDWLTH,SUCDPAED,SUCDKNOW
YEAR,,,,
2003.0,0,0,0,0
2004.0,0,0,0,0
2005.0,0,0,0,0
2006.0,0,0,0,0
2007.0,0,0,0,0
2008.0,0,0,0,0
2009.0,1597,1595,1597,1595
2010.0,0,0,0,0
2011.0,0,0,0,0


KIDSOL06 value labels
-8.0: DK
-1.0: IAP
1.0: 훨씬 좋아질 것
2.0: 약간 좋아질 것
3.0: 차이가 거의 없을 것
4.0: 약간 나빠질 것
5.0: 훨씬 나빠질 것
8.0: DK/Refusal
Aggregated summary tables saved to: ../outputs/tables


## 11. 다음 단계 메모

이 노트북에서 변수 존재 여부와 연도별 응답 수를 확인한 뒤, 실제 분석 노트북에서는 사용할 연도와 변수 범위를 정하고 가중치(`FINALWT`) 적용 방식, 결측값 처리 방식, 집계표 생성 방식을 결정하면 됩니다.